# 04 — Duplicate Analysis

## Objective

Identify exact duplicate images and rank visually similar image pairs for manual review.

## Motivation

Duplicate and near-duplicate images can inflate dataset size and create leakage across splits. Exact duplicates are established with SHA-256, while perceptual similarity is treated only as a review signal because no universal threshold is reliable.

## Inputs

- `curation/outputs/01-dataset-audit/inventory.csv`
- Images from the configured image directory.

## Outputs

- `curation/outputs/04-duplicate-analysis/exact_duplicates.csv`
- `curation/outputs/04-duplicate-analysis/duplicate_review_candidates.csv`
- `curation/outputs/04-duplicate-analysis/manifest.yaml`

Intermediate perceptual hashes are not persisted.


In [ ]:
# Environment-specific setup
import pathlib
import sys

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../../')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

ROOT = base_folder.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from curation.common import load_config, prepare_dataset, stage_output_dir

CONFIG = load_config(ROOT)
DATASET = prepare_dataset(ROOT)
IMAGES_DIR = DATASET["images_dir"]
ANNOTATIONS_DIR = DATASET["annotations_dir"]

from curation.common import write_manifest
import heapq
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from skimage.metrics import structural_similarity as ssim

STAGE = "04-duplicate-analysis"
STAGE_DIR = stage_output_dir(STAGE, ROOT)
INVENTORY_PATH = stage_output_dir("01-dataset-audit", ROOT, create=False) / "inventory.csv"
if not INVENTORY_PATH.is_file():
    raise FileNotFoundError("Run 01-dataset-audit.ipynb first.")

In [ ]:
inventory = pd.read_csv(INVENTORY_PATH)
readable = inventory[inventory["readable"]].copy()
exact = readable.groupby("sha256").filter(lambda group: len(group) > 1).sort_values(["sha256", "relative_path"])
exact_path = STAGE_DIR / "exact_duplicates.csv"
exact.to_csv(exact_path, index=False)

duplicate_config = CONFIG["audits"]["duplicate"]
resize = int(duplicate_config["phash_resize"])
low_size = int(duplicate_config["phash_low_frequency_size"])

def phash255(path):
    with Image.open(path).convert("L") as image:
        pixels = np.asarray(image.resize((resize, resize), Image.Resampling.LANCZOS), dtype=np.float32)
    low = cv2.dct(pixels)[:low_size, :low_size].ravel()[1:]
    return int("".join("1" if value > np.median(low) else "0" for value in low), 2)

hashes = [(row.relative_path, row.sha256, phash255(IMAGES_DIR / row.relative_path)) for row in readable.itertuples(index=False)]

In [ ]:
top_k = int(duplicate_config["top_k_pairs"])
heap = []
for left in range(len(hashes)):
    for right in range(left + 1, len(hashes)):
        distance = (hashes[left][2] ^ hashes[right][2]).bit_count()
        item = (-distance, left, right)
        if len(heap) < top_k:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)

ssim_size = int(duplicate_config["ssim_resize"])
rows = []
for negative_distance, left, right in sorted(heap, reverse=True):
    path_a, sha_a, _ = hashes[left]
    path_b, sha_b, _ = hashes[right]
    with Image.open(IMAGES_DIR / path_a).convert("L") as image_a, Image.open(IMAGES_DIR / path_b).convert("L") as image_b:
        a = np.asarray(image_a.resize((ssim_size, ssim_size), Image.Resampling.LANCZOS))
        b = np.asarray(image_b.resize((ssim_size, ssim_size), Image.Resampling.LANCZOS))
    rows.append({"image_a": path_a, "image_b": path_b, "phash_hamming_distance": -negative_distance, "ssim_256_gray": float(ssim(a, b, data_range=255)), "same_sha256": sha_a == sha_b})

candidates = pd.DataFrame(rows).sort_values(["phash_hamming_distance", "ssim_256_gray"], ascending=[True, False]).reset_index(drop=True)
candidates.insert(0, "rank", np.arange(1, len(candidates) + 1))
candidates_path = STAGE_DIR / "duplicate_review_candidates.csv"
candidates.to_csv(candidates_path, index=False)
display(candidates.head(20))

In [ ]:
write_manifest(
    STAGE,
    "04-duplicate-analysis.ipynb",
    inputs={"inventory": INVENTORY_PATH},
    parameters={"hash": "sha256", "phash_bits": low_size * low_size - 1, "phash_resize": resize, "top_k_pairs": top_k, "ssim_resize": ssim_size},
    artifacts=[exact_path, candidates_path],
    summary={"exact_duplicate_images": int(len(exact)), "exact_duplicate_groups": int(exact.groupby("sha256").ngroups), "review_candidate_pairs": int(len(candidates))},
    repo_root=ROOT,
)

## Inspect a Duplicate Candidate

Use the zero-based row index from `duplicate_review_candidates.csv` to display both images side by side. The function also prints the complete CSV row, including pHash distance and SSIM.

In [ ]:
def show_duplicate_candidate(csv_index):
    """Display the duplicate pair at a CSV row.

    Args:
        csv_index: Zero-based row index in duplicate_review_candidates.csv.
    """
    import matplotlib.pyplot as plt

    rows = pd.read_csv(candidates_path)
    if not isinstance(csv_index, (int, np.integer)):
        raise TypeError("csv_index must be an integer.")
    if csv_index < 0 or csv_index >= len(rows):
        raise IndexError(f"csv_index must be between 0 and {len(rows) - 1}.")

    candidate = rows.iloc[int(csv_index)]
    image_paths = [
        IMAGES_DIR / candidate["image_a"],
        IMAGES_DIR / candidate["image_b"],
    ]

    figure, axes = plt.subplots(1, 2, figsize=(12, 5))
    for axis, image_path, column in zip(axes, image_paths, ["image_a", "image_b"]):
        with Image.open(image_path).convert("RGB") as image:
            axis.imshow(image)
        axis.set_title(f"{column}: {image_path.name}")
        axis.axis("off")

    figure.suptitle(
        f"CSV row {csv_index} | pHash distance={candidate['phash_hamming_distance']} | "
        f"SSIM={candidate['ssim_256_gray']:.4f}"
    )
    plt.tight_layout()
    plt.show()

    print(candidate.to_string())
    return candidate

# Example:
show_duplicate_candidate(0)

In [ ]:
for i in range(10):
    show_duplicate_candidate(i)